# PhotoGraphiQML — All Tutorials (Colab)

This notebook merges the 44 scripts in `examples/tutorials/`. Each tutorial has two cells:

* a **markdown cell** explaining what the code does, plus the relevant maths and algorithms;
* a **code cell** that is *independently executable*. It makes its own imports and asserts its expected result.

Run the **Setup** cell once, then run any tutorial cell in any order.

**Layout.** Tutorials 01–30 cover the *logical* qubit MuTA (Multiple Triangle Ansatz): graph-state MBQC, training, kernels, and diagnostics. Tutorials 31–44 cover the *physical* finite-energy GKP bridge built on PhotoGraphiQ and the Piquasso Fock backend.

## Setup

In [ ]:
# Setup: install the unified PhotoGraphiQ repository, which ships photographiqml beside
# photographiq (and pulls in piquasso, numpy, scipy, networkx). In Colab this cell clones
# the repository; locally, start Jupyter from the repository root or from examples/ml/.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/chinmoybiswasdeep/PhotoGraphiQ.git"
REPOSITORY_BRANCH = "main"
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    repository_root = Path("/content/PhotoGraphiQ")
    if not repository_root.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPOSITORY_BRANCH,
                REPOSITORY_URL,
                str(repository_root),
            ],
            check=True,
        )
else:
    current = Path.cwd().resolve()
    repository_root = current.parents[1] if current.parts[-2:] == ("examples", "ml") else current
    if not (repository_root / "pyproject.toml").exists():
        raise FileNotFoundError("Run locally from the PhotoGraphiQ repository root.")

os.chdir(repository_root)
# Editable install; tutorial 20 optionally uses MentPy via ".[ml-validation]".
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[ml-experiments,visualization]"], check=True
)
# An editable install is only picked up by new processes, so expose src/ to this kernel too.
sys.path.insert(0, str(repository_root / "src"))

import photographiq  # noqa: E402
import photographiqml  # noqa: E402

print("photographiqml", photographiqml.__version__, "| photographiq", photographiq.__version__)

## 01 · What is MB-QML?

**What the code does.** It builds a single-wire MuTA (Multiple Triangle Ansatz), feeds in $|0\rangle$, uses the default angles, and checks that the output distribution is $[1, 0]$.

**Maths.** Measurement-based quantum computing (MBQC) starts from a *graph state*
$$|G\rangle = \prod_{(a,b)\in E} CZ_{ab}\,|+\rangle^{\otimes V}.$$
If you measure qubit $a$ of a wire in the XY-plane basis $|\pm_\alpha\rangle = (|0\rangle \pm e^{i\alpha}|1\rangle)/\sqrt2$, the logical state is teleported to its neighbour and transformed by
$$J(\alpha) = \tfrac{1}{\sqrt2}\begin{pmatrix}1 & e^{i\alpha}\\ 1 & -e^{i\alpha}\end{pmatrix} = H\,R_z(\alpha),$$
with a random Pauli by-product $X^{s}$ that depends on the outcome bit $s$. A wire with four measurements implements $J(\alpha_4)J(\alpha_3)J(\alpha_2)J(\alpha_1)$. With every $\alpha_k=0$ this is $H^4 = I$.

**Algorithm.** Measure the nodes in the order given by the graph's *flow*, apply the accumulated Pauli frame correction, then read the output wire. In MB-QML the measurement angles $\alpha$ are the trainable parameters.

In [1]:
import numpy as np

import photographiqml as pqml

m = pqml.MuTA(1)
r = m.run([1, 0])
assert np.allclose(r.probabilities, [1, 0])
print(r.probabilities.round(6))

[1. 0.]


## 02 · MuTA from the paper

**What the code does.** It builds a two-wire, one-column MuTA, sets the base-centre angle `alpha.w1.c1` $=\pi/2$, and runs the input $|00\rangle$.

**Maths.** In the paper's Table I, the triangle tip couples the two wires, and the measurement at the base centre implements an Ising-XX interaction:
$$U(\phi) = \exp\!\big(+i\tfrac{\phi}{2}X\otimes X\big) = \cos\tfrac\phi2\,I + i\sin\tfrac\phi2\,XX.$$
Acting on $|00\rangle$ gives $\cos\frac\phi2|00\rangle + i\sin\frac\phi2|11\rangle$. When $\phi=\pi/2$ this is a maximally entangled Bell-type state with $P(00)=P(11)=\tfrac12$.

**Note.** These are logical MBQC angles, not homodyne settings. The sign convention is $e^{+i\phi XX/2}$.

In [ ]:
import numpy as np

import photographiqml as pqml

m = pqml.MuTA(2, one_column=True)
r = m.run([1, 0, 0, 0], {"alpha.w1.c1": np.pi / 2})
assert np.allclose(r.probabilities, [0.5, 0, 0, 0.5])
print(r.probabilities.round(6))

## 03 · First MuTA model

**What the code does.** It prints the model summary and checks that a two-wire, one-column MuTA has $8$ trainable parameters.

**Counting.** Each wire contributes 5 resource sites: 1 input followed by 4 measured columns. Only the 4 measured columns carry angles, so there are $2\times4=8$ angles.
CZ edges: $2\times4$ along the wires plus the 2 edges from the triangle tip, giving 10. The *causal dependencies* are the edges of the flow's partial order. They state which measurement outcome feeds forward into which later angle (via $\alpha \to (-1)^{s_x}\alpha + s_z\pi$).

**Conventions.** Wire 0 is the most significant tensor factor, so a basis index is $|w_0 w_1\cdots\rangle$.

In [ ]:
import photographiqml as pqml

m = pqml.MuTA(2, one_column=True)
assert m.n_parameters == 8
print(m.summary())

## 04 · Triangle neuron anatomy

**What the code does.** It builds the bare `TriangleNeuron(2)` resource graph and checks that the tip $(0,1)$ connects to both ends $(1,0)$ and $(1,2)$ of a three-site base.

**Graph theory.** Node labels are $(\text{wire}, \text{column})$. The triangle is the cycle $(0,1)-(1,0)-(1,1)-(1,2)-(0,1)$. Despite its name, this "triangle" is a *4-cycle*, and the whole resource graph is **bipartite** (colour nodes by the parity of wire+column).
The same edges, sorted, form the CZ set $E$ in $|G\rangle=\prod_E CZ|+\rangle^{\otimes V}$.

In [ ]:
import photographiqml as pqml

cell = pqml.TriangleNeuron(2)
assert cell.graph.has_edge((0, 1), (1, 0))
assert cell.graph.has_edge((0, 1), (1, 2))
print(sorted(cell.graph.edges))

## 05 · Multi-wire MuTA

**What the code does.** It builds `MuTA(3, 1)` (3 wires, 1 requested layer) and checks its paper depth and node count.

**Maths.** For $n$ wires, a *full* layer rotates the triangle tip through every wire, so it unrolls into $n$ paper layers. Here $n=3$ gives `paper_depth == 3`.
Each paper layer adds 4 sites per wire, so
$$|V| = n\,(1 + 4\cdot\text{paper\_depth}) = 3\,(1+12) = 39,\qquad \#\theta = 3\cdot4\cdot3 = 36.$$
The constructor only accepts tip placements that preserve *gflow*, which is the condition for deterministic MBQC. Adding tips arbitrarily can break it.

In [ ]:
import photographiqml as pqml

m = pqml.MuTA(3, 1)
assert m.paper_depth == 3
assert len(m.graph) == 39
print(m.summary())

## 06 · Multi-layer MuTA

**What the code does.** It stacks two one-column layers on two wires and checks the graph size and the output node labels.

**Maths.** Composing layers identifies the output sites of layer $\ell$ with the input sites of layer $\ell+1$, so the total unitary is $U = U_2 U_1$, and
$$|V| = n(1+4L) = 2(1+8) = 18,\qquad \text{outputs} = \{(w, 4L)\} = \{(0,8),(1,8)\}.$$
Semantic $(\text{wire},\text{column})$ coordinates stay stable when layers are stacked. Integer node labels (MentPy-style) do not.

In [ ]:
import photographiqml as pqml

m = pqml.MuTA(2, 2, one_column=True)
assert len(m.graph) == 18
assert m.output_nodes == ((0, 8), (1, 8))
print(m.output_nodes)

## 07 · Trainable measurement angles

**What the code does.** It freezes the angle `alpha.w0.c3` at $0$, lists the remaining trainable names, and then unfreezes it.

**Maths.** The model is a map $\theta\in\mathbb{R}^P \mapsto U(\theta)$. Freezing coordinate $k$ at value $c$ restricts this to the sub-family $\{U(\theta): \theta_k=c\}$ and lowers $P$ from 4 to 3. Names follow `alpha.w{wire}.c{column}`.
Frozen values are metadata on the parameter, so a call to `run` cannot override them.

In [ ]:
import photographiqml as pqml

m = pqml.MuTA(1)
m.freeze("alpha.w0.c3", 0)
assert m.n_parameters == 3
print(list(m.trainable_parameters()))
m.unfreeze("alpha.w0.c3")
assert m.n_parameters == 4

## 08 · Classical feature encoding

**What the code does.** It encodes the feature vector $x=(0,\pi)$ into a 2-qubit state.

**Maths.** This is product $R_y$ angle encoding with one qubit per feature:
$$|\psi(x)\rangle = \bigotimes_j R_y(x_j)|0\rangle = \bigotimes_j\big(\cos\tfrac{x_j}{2}|0\rangle + \sin\tfrac{x_j}{2}|1\rangle\big).$$
For $x=(0,\pi)$ this gives $|0\rangle\otimes|1\rangle=|01\rangle$, so the probabilities are $[0,1,0,0]$. Features are read as radians and nothing is rescaled implicitly.

In [ ]:
import numpy as np

from photographiqml.models import angle_encode

s = angle_encode([0, np.pi])
assert np.allclose(abs(s) ** 2, [0, 1, 0, 0])
print(abs(s).round(6))

## 09 · Quantum state inputs

**What the code does.** It feeds the Bell state $|\Phi^+\rangle=(|00\rangle+|11\rangle)/\sqrt2$ directly into a full two-wire MuTA with its default angles and checks that it comes out unchanged.

**Maths.** At the default (all-zero) angles every wire realises $H^4=I$, so $U=I$ and the fidelity is
$$F = |\langle\Phi^+|U|\Phi^+\rangle|^2 = 1.$$
This also shows that a quantum input bypasses classical encoding and is transformed coherently, keeping its entanglement.

In [ ]:
import numpy as np

import photographiqml as pqml

s = np.array([1, 0, 0, 1]) / np.sqrt(2)
r = pqml.MuTA(2).run(s)
assert abs(np.vdot(s, r.state)) ** 2 > 1 - 1e-12
print(r.probabilities.round(6))

## 10 · Gate learning

**What the code does.** It trains a one-wire MuTA to reproduce the target gate $V=e^{\,0.4 i X}$ from 7 random training states.

**Maths.** The training set is $\{|s_i\rangle\}$, drawn Haar-randomly by normalising complex Gaussian vectors. The targets are $|t_i\rangle = V|s_i\rangle$ (the code writes `s @ V.T` for a batch of row vectors). The loss is the average infidelity
$$\mathcal{L}(\theta) = 1 - \frac1N\sum_{i=1}^N \big|\langle t_i|U(\theta)|s_i\rangle\big|^2 .$$
**Algorithm.** Adam ($\eta=0.05$, 100 epochs) with central finite-difference gradients, starting from $\theta=0$. Any single-qubit unitary is reachable as $J(\alpha_4)\cdots J(\alpha_1)$ (an Euler decomposition), so the loss can go to $\approx 0$.

In [ ]:
import numpy as np
from scipy.linalg import expm

import photographiqml as pqml
from photographiqml.logical import X
from photographiqml.models import haar_states, infidelity

m = pqml.MuTA(1)
s = haar_states(1, 7, 42)
t = s @ expm(0.4j * X).T


def loss(p):
    return infidelity(m.run_batch(s, p), t)


p, h = pqml.Trainer(epochs=100).fit(loss, np.zeros(4))
assert h.losses[-1] < 1e-4
print(round(h.losses[-1], 6))

## 11 · State classification via quantum Fisher information

**What the code does.** It computes the quantum Fisher information (QFI) of the 2-qubit GHZ/Bell state for the collective phase generator $H=(Z_1+Z_2)/2$.

**Maths.** For a pure probe $|\psi\rangle$ and the unitary family $e^{-i\theta H}$,
$$F_Q = 4\,\mathrm{Var}_\psi(H) = 4\big(\langle H^2\rangle - \langle H\rangle^2\big).$$
For $|\Phi^+\rangle$: $\langle H\rangle = 0$ and $H^2=1$ on $\mathrm{span}\{|00\rangle,|11\rangle\}$, so $F_Q=4=N^2$. This is the Heisenberg limit. A product state reaches only the standard quantum limit $N=2$.
Thresholding $F_Q > N$ can therefore label states as "metrologically useful". This is what the paper's state-classification task learns.

In [ ]:
import numpy as np

from photographiqml.diagnostics import pure_qfi
from photographiqml.logical import Z

H = (np.kron(Z, np.eye(2)) + np.kron(np.eye(2), Z)) / 2
s = np.array([1, 0, 0, 1]) / np.sqrt(2)
qfi = pure_qfi(s, H)
assert np.isclose(qfi, 4)
print(qfi)

## 12 · Classical classification

**What the code does.** It trains `MuTAClassifier` on four 1-D points, so that $x\approx0 \mapsto 1$ and $x\approx\pi\mapsto 0$, and then predicts two new points.

**Model.**
$$x \xrightarrow{R_y} |\psi(x)\rangle \xrightarrow{U(\theta)} f(x)=\langle Z_0\rangle,\qquad p(y=1|x)=\sigma\big(w f(x)+b\big).$$
**Loss.** Binary cross-entropy written in a numerically stable form: $\ell = \log(1+e^{z}) - y z$ with $z=w f + b$. All parameters ($\theta$, $w$, $b$) are trained with Adam for 60 epochs.
Accuracy on the training set is only an API smoke test. Real studies need held-out data and baselines.

In [ ]:
import photographiqml as pqml

X = [[0], [0.1], [3.0], [3.14]]
y = [1, 1, 0, 0]
c = pqml.MuTAClassifier(pqml.MuTA(1), trainer=pqml.Trainer(epochs=60))
c.fit(X, y)
assert c.score(X, y) == 1
print(c.predict([[0.05], [3.1]]))

## 13 · Quantum instrument learning

**What the code does.** It wraps a MuTA in a `QuantumInstrumentModel`, which measures wire 0 in $Z$ after the circuit, and returns each outcome with its probability and post-measurement state.

**Maths.** A quantum instrument is a set of completely positive maps $\{\mathcal{E}_k\}$ with $\sum_k\mathcal{E}_k$ trace-preserving. Here
$$p_k = \|(\langle k|_0\otimes I)\,U|\psi\rangle\|^2,\qquad |\psi_k\rangle = \frac{(\langle k|_0\otimes I)U|\psi\rangle}{\sqrt{p_k}}.$$
For a Bell input and identity $U$: $p_0=p_1=\tfrac12$, and $\sum_k p_k=1$ is checked.

In [ ]:
import numpy as np

import photographiqml as pqml

m = pqml.QuantumInstrumentModel(pqml.MuTA(2, one_column=True))
b = m.run(np.array([1, 0, 0, 1]) / np.sqrt(2))
assert np.isclose(sum(x.probability for x in b), 1)
print([(x.outcome, round(x.probability, 6)) for x in b])

## 14 · MuTA quantum kernel

**What the code does.** It builds the paper's Eq. 5 feature map with a two-wire MuTA, computes the Gram matrix for 3 points, and checks that it is positive semidefinite (PSD).

**Feature map.** For $x=(x_0,x_1)$ the angles are $\alpha_{w0,c0}=\alpha_{w0,c2}=x_0$, $\alpha_{w1,c0}=\alpha_{w1,c2}=x_1$, and the entangling angle is $\alpha_{w1,c1}=\cos x_0\cos x_1$. This gives $|\phi(x)\rangle = U(x)|00\rangle$.

**Kernel.**
$$k(x,x') = |\langle\phi(x)|\phi(x')\rangle|^2 = \mathrm{Tr}\big[\rho(x)\rho(x')\big].$$
This is a Hilbert–Schmidt inner product of density matrices, so $K_{ij}=k(x_i,x_j)$ is PSD (Mercer). Its minimum eigenvalue should be $\ge 0$ up to round-off. Such a kernel can be plugged into an SVM.

In [ ]:
import photographiqml as pqml

k = pqml.MuTAKernel()
X = [[0, 0], [1, 0], [0, 1]]
d = k.diagnostics(X)
assert d["minimum_eigenvalue"] > -1e-10
print(k.gram_matrix(X).round(4))

## 15 · Training with Adam

**What the code does.** It minimises $f(p)=\|p-0.3\|^2$ from $p_0=(1,-1)$ using `Trainer`'s default Adam optimiser.

**Algorithm (Adam).** With gradient $g_t$:
$$m_t=\beta_1 m_{t-1}+(1-\beta_1)g_t,\quad v_t=\beta_2 v_{t-1}+(1-\beta_2)g_t^2,$$
$$\hat m_t=\frac{m_t}{1-\beta_1^t},\quad \hat v_t=\frac{v_t}{1-\beta_2^t},\quad p_{t+1}=p_t-\eta\,\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.$$
Each `fit` resets $m, v$. Without a supplied gradient, $g_t$ comes from central finite differences (next tutorial).

In [ ]:
import numpy as np

import photographiqml as pqml

p, h = pqml.Trainer(epochs=180).fit(lambda p: np.sum((p - 0.3) ** 2), [1, -1])
assert np.linalg.norm(p - 0.3) < 0.001
print(p.round(4))

## 16 · Finite-difference validation

**What the code does.** It numerically differentiates $f(x)=\sum x_i^2$ at $(0.3,-0.2)$ and compares the result with the exact gradient $2x$.

**Maths.** The central difference is
$$\partial_i f(x)\approx\frac{f(x+h e_i)-f(x-h e_i)}{2h}.$$
Its truncation error is $O(h^2)$ (the odd Taylor terms cancel) and its round-off error is $O(\varepsilon_{\text{mach}}/h)$. The two are balanced at $h\sim\varepsilon^{1/3}$. For a quadratic the truncation error is exactly zero.
For logical MuTA the exact alternative is the parameter-shift rule $\partial_\alpha\langle O\rangle = \tfrac12[\langle O\rangle_{\alpha+\pi/2}-\langle O\rangle_{\alpha-\pi/2}]$.

In [ ]:
import numpy as np

from photographiqml.training import finite_difference

p = np.array([0.3, -0.2])
g = finite_difference(lambda x: np.sum(x**2), p)
assert np.allclose(g, 2 * p, atol=1e-8)
print(g.round(6))

## 17 · Supplying an analytic gradient (L-BFGS)

**What the code does.** It minimises $f(x)=\|x\|^2$ with L-BFGS and passes the exact gradient $\nabla f=2x$ as a callable.

**Algorithm (L-BFGS).** A quasi-Newton method. It keeps the last $m$ pairs $s_k=x_{k+1}-x_k$, $y_k=\nabla f_{k+1}-\nabla f_k$ and builds an implicit inverse-Hessian approximation $H_k$ with the two-loop recursion. The step direction is $d_k=-H_k\nabla f_k$, followed by a line search.
On a convex quadratic it converges in very few iterations. No autodiff is claimed here: a gradient callable only helps if it is correct.

In [ ]:
import numpy as np

import photographiqml as pqml

p, h = pqml.Trainer("lbfgs", epochs=20).fit(
    lambda x: np.sum(x * x), [1, 2], gradient=lambda x: 2 * x
)
assert np.linalg.norm(p) < 1e-8
print(p.round(6))

## 18 · Finite-squeezing noise of a Gaussian wire (PhotoGraphiQ)

**What the code does.** It uses PhotoGraphiQ's Gaussian `wire_channel` for 4 teleportation steps, all at zero shear, with a finite squeezing parameter of $1.0$, and returns the symplectic part $S$ and the added noise $N$.

**Maths.** A Gaussian channel acts on quadrature covariance matrices as
$$\sigma \mapsto S\sigma S^T + N .$$
With ideal infinite squeezing, CV teleportation is $S=I$ (four zero-shear steps compose to the identity) and $N=0$. Finite squeezing $r$ adds noise $\propto e^{-2r}$ at every step, so $N \succ 0$. The code asserts $S=I$ and $\mathrm{Tr}N>0$.
This is a channel diagnostic, not a GKP gate.

In [ ]:
import numpy as np

from photographiq.gaussian import wire_channel

S, N = wire_channel([0] * 4, 1.0)
assert np.allclose(S, np.eye(2))
assert np.trace(N) > 0
print(N.round(6))

## 19 · GKP logical MuTA

**What the code does.** It requests a MuTA with `representation="gkp"`. It then shows that `GKPBridge.logical_target` returns the *ideal logical* target, and that direct physical execution is refused with `NotImplementedError`.

**Concept.** A GKP qubit encodes $|0_L\rangle,|1_L\rangle$ as combs of peaks in the position quadrature at $q = 2n\sqrt\pi$ and $(2n+1)\sqrt\pi$. Running arbitrary logical XY-plane measurements on such a code needs a measurement-synthesis instrument and a decoder, which do not exist yet. The library therefore fails explicitly and does not silently return logical numbers labelled as physical ones.

In [ ]:
import photographiqml as pqml

m = pqml.MuTA(1, representation="gkp")
r = pqml.GKPBridge().logical_target(m, [1, 0])
assert r.representation == "logical"
try:
    m.run([1, 0])
except NotImplementedError:
    print("Legacy resource-only execution is unsupported; ideal target remains logical")
else:
    raise AssertionError("Unexpected physical execution")

## 20 · MentPy cross-validation (optional)

**What the code does.** If the optional `mentpy` package is installed, it compares PhotoGraphiQML's logical density matrix with MentPy's reference simulator for random angles. Otherwise it prints a notice.

**Maths.** The error is a matrix-norm distance $\|\rho_{\text{pqml}}-\rho_{\text{MentPy}}\|$ between the two output density matrices, which should be below $10^{-10}$. Using $\rho$ rather than $|\psi\rangle$ removes the global phase.
To enable it on Colab, run `!pip -q install mentpy==0.1.0a15` first.

In [ ]:
import importlib.util

import photographiqml as pqml

if importlib.util.find_spec("mentpy"):
    from photographiqml.validation import compare_mentpy

    m = pqml.MuTA(2, one_column=True)
    error = compare_mentpy(m, [1, 0, 0, 0], m.initialize(2))
    assert error < 1e-10
    print("Logical density agreement:", error < 1e-10)
else:
    print("Optional MentPy reference not installed")

## 21 · Expressivity vs depth

**What the code does.** It checks that `MuTA(2,1)` and `MuTA(2,2)` produce the same unitary for the same (random) parameters. The second layer's extra angles default to zero, which is an identity layer.

**Maths.** Write $\mathcal{U}_L=\{U_L(\theta)\}$ for the family reachable at depth $L$. An all-zero layer gives $J(0)^4=H^4=I$, so
$$U_{L+1}(\theta, 0) = I\cdot U_L(\theta)\ \Rightarrow\ \mathcal{U}_L\subseteq\mathcal{U}_{L+1}.$$
This is *weak* inclusion: going deeper never hurts expressivity. It does not imply a strictly larger family at every depth.

In [ ]:
import numpy as np

import photographiqml as pqml

a, b = pqml.MuTA(2, 1), pqml.MuTA(2, 2)
p = a.initialize(3)
assert np.allclose(a.unitary(p), b.unitary(p))
print("Identity extension verified")

## 22 · Tunable entanglement

**What the code does.** It sets the entangling angle to $\phi=0.7$ and checks that the output's concurrence equals $|\sin\phi|$.

**Maths.** For a pure 2-qubit state $a|00\rangle+b|01\rangle+c|10\rangle+d|11\rangle$, the concurrence is
$$C = 2|ad-bc|.$$
For $e^{i\phi XX/2}|00\rangle=\cos\frac\phi2|00\rangle+i\sin\frac\phi2|11\rangle$ this gives $C=2|\cos\frac\phi2\sin\frac\phi2|=|\sin\phi|$. So $\phi$ tunes the entanglement continuously from product ($\phi=0,\pi$) to maximal ($\phi=\pi/2$).

In [ ]:
import numpy as np

import photographiqml as pqml
from photographiqml.diagnostics import concurrence

m = pqml.MuTA(2, one_column=True)
phi = 0.7
c = concurrence(m.run([1, 0, 0, 0], {"alpha.w1.c1": phi}).state)
assert np.isclose(c, abs(np.sin(phi)))
print(round(c, 6))

## 23 · Bias engineering

**What the code does.** It builds a two-wire MuTA *without* inter-wire connections (`connections=()`), so the wires stay separate, and then freezes and unfreezes one angle.

**Concept.** The inductive bias is set by (i) the **graph topology**, since no tip means no entangling edge and hence $U = U_0\otimes U_1$, and (ii) **angle constraints**, since frozen angles define a sub-family.
Edge count: $2\times4=8$ wire edges and no tip edges.

In [ ]:
import photographiqml as pqml

m = pqml.MuTA(2, one_column=True, connections=())
assert m.graph.number_of_edges() == 8
m.freeze("alpha.w0.c1", 0)
assert m.n_parameters == 7
m.unfreeze("alpha.w0.c1")
assert m.n_parameters == 8
print("Topology and angle constraints are explicit")

## 24 · Fisher information (local expressivity)

**What the code does.** It computes the pure-state QFI matrix of a one-wire MuTA at random parameters and reports its numerical rank.

**Maths.** With $|\partial_i\psi\rangle$ obtained by central differences,
$$F_{ij} = 4\,\mathrm{Re}\Big[\langle\partial_i\psi|\partial_j\psi\rangle - \langle\partial_i\psi|\psi\rangle\langle\psi|\partial_j\psi\rangle\Big].$$
The subtracted term projects out the unobservable global phase, and $F\succeq0$. Its rank counts the locally independent directions in state space. A single-qubit pure state lives on the Bloch sphere (2 real dimensions), so the rank is at most 2 even though there are 4 angles.

In [ ]:
import numpy as np

import photographiqml as pqml
from photographiqml.expressivity import state_fisher

m = pqml.MuTA(1)
F = state_fisher(m, [1, 0], list(m.initialize(1).values()))
assert np.linalg.eigvalsh(F).min() > -1e-9
print("Local rank:", np.linalg.matrix_rank(F, tol=1e-8))

## 25 · Model serialization

**What the code does.** It freezes one angle, saves the model to JSON in a temporary directory, reloads it, and checks that the unitaries match: $\|U-U_{\text{restored}}\|\approx0$.

**Concept.** The JSON stores the topology (nodes, CZ edges, flow), the parameter values, and the frozen metadata. Loading it runs no code, unlike `pickle`, so it is safe for untrusted files. Optimizer state and classifier heads are *not* saved.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

import photographiqml as pqml

m = pqml.MuTA(1)
m.freeze("alpha.w0.c1", 0.3)
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "model.json"
    m.save(p)
    restored = pqml.MuTA.load(p)
    assert np.allclose(m.unitary(), restored.unitary())
print("Logical model round trip verified")

## 26 · Custom observable readout

**What the code does.** It evaluates $\langle Z\rangle$ on the output of a one-wire MuTA with input $|0\rangle$.

**Maths.** For a Hermitian $O$ of matching dimension (in wire order),
$$\langle O\rangle = \langle\psi|O|\psi\rangle \in \mathbb{R}.$$
With $O=\mathrm{diag}(1,-1)=Z$ and $|\psi\rangle=|0\rangle$, this gives $\langle Z\rangle=1$. The library checks that $O$ is Hermitian and has the right dimensions.

In [ ]:
import numpy as np

import photographiqml as pqml

m = pqml.MuTA(1)
r = m.run([1, 0])
value = r.expectation(np.diag([1, -1]))
assert np.isclose(value, 1)
print(value)

## 27 · Custom loss

**What the code does.** It passes an arbitrary user-defined scalar loss, the mean squared error to a target vector $\mathrm{MSE}(p)=\frac1n\sum_i(p_i-t_i)^2$, to `Trainer("lbfgs")`.

**Concept.** `Trainer.fit` accepts any deterministic $\mathcal{L}:\mathbb{R}^P\to\mathbb{R}$. In practice $\mathcal{L}$ would wrap model predictions, e.g. $\mathcal{L}(\theta) = \frac1N\sum_i (\langle O\rangle_{\theta,x_i}-y_i)^2$. The minimiser here is $p^\star=t$ with $\mathcal{L}(p^\star)=0$.

In [ ]:
import numpy as np

import photographiqml as pqml

target = np.array([0.5, -0.4])


def loss(p):
    return float(np.mean((p - target) ** 2))


p, h = pqml.Trainer("lbfgs", epochs=20).fit(loss, [0, 0])
assert loss(p) < 1e-10
print(round(loss(p), 8))

## 28 · Training callbacks

**What the code does.** It registers a callback that records the step index and $\|p\|$ during a 3-epoch Adam run of $f(p)=p^2$.

**Concept.** The callback gets called as `callback(params_copy, history)` at the initial point *and* after every epoch, giving $1+\text{epochs}=4$ calls. The step index is $|\text{history.losses}|-1$. Use callbacks for logging, diagnostics, or checkpointing.

In [ ]:
import numpy as np

import photographiqml as pqml

seen = []


def callback(p, h):
    seen.append((len(h.losses) - 1, float(np.linalg.norm(p))))


p, h = pqml.Trainer(epochs=3).fit(lambda p: np.sum(p * p), [1], callback=callback)
assert len(seen) == 4
print([step for step, norm in seen])

## 29 · CV teleportation cell (CVMuTA proposal)

**What the code does.** It checks a continuous-variable (CV) teleportation step matrix from PhotoGraphiQ.

**Maths.** One CV cluster-state teleportation step acts on the quadratures $(q,p)$ as
$$T(k) = \begin{pmatrix}-k & -1\\ 1 & 0\end{pmatrix},\qquad k=\cot\theta .$$
At zero shear $k=0$, $T(0)$ is a $90^\circ$ phase-space rotation (a Fourier transform), so $T(0)^4=I$. This is the CV analogue of $H^4=I$ in tutorial 01. It could serve as an ingredient for a future CVMuTA cell, which remains a proposal.

In [ ]:
import numpy as np

from photographiq.gaussian import teleportation_matrix

T = teleportation_matrix(0)
assert np.allclose(np.linalg.matrix_power(T, 4), np.eye(2))
print("CV cell ingredient verified; CVMuTA is a proposal")

## 30 · Comparing qubit, GKP and CV representations

**What the code does.** It runs `GKPBridge.diagnostics()` with Fock cutoff 24 and checks the *captured weights*: how much of each finite-energy GKP codeword's norm survives truncation to $n<24$ photons.

**Maths.** For a codeword $|\mu_L\rangle$ (unnormalised before truncation):
$$w_\mu = \sum_{n<\text{cutoff}} |\langle n|\mu_L\rangle|^2 \Big/ \langle\mu_L|\mu_L\rangle \in (0,1].$$
Values near 1 mean the truncation is adequate. The report also states `physical_muta_validated = False`. Logical fidelity, codeword capture, and Gaussian channel noise answer *different* questions, so each metric has to stay attached to its own representation.

In [ ]:
import numpy as np

import photographiqml as pqml

r = pqml.GKPBridge(cutoff=24, grid_points=2049).diagnostics()
assert not r["physical_muta_validated"]
assert all(0 < w <= 1.000001 for w in r["captured_weights"])
print("GKP captured weights:", np.round(r["captured_weights"], 5))

## 31 · PhotoGraphiQ 0.3.1 encoded interface

**What the code does.** It checks the PhotoGraphiQ interface contract. It then encodes the logical $|0\rangle$ into a finite-energy GKP Fock vector and confirms that the vector is normalised.

**Maths.** A finite-energy GKP codeword is
$$|\mu_L\rangle \propto \sum_{s=-\text{peaks}}^{\text{peaks}} e^{-\kappa^2 x_s^2/2}\,\hat D(x_s)\,|\text{sq}_\Delta\rangle,\qquad x_s=(2s+\mu)\sqrt\pi,$$
i.e. Gaussian peaks of width $\Delta$ under an envelope of width $1/\kappa$, expanded in the Fock basis up to the cutoff. Finite codewords overlap slightly ($\langle0_L|1_L\rangle\neq0$), so the normalised encoding $c_0|0_L\rangle+c_1|1_L\rangle$ is not an exact isometry.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
import numpy as np

from photographiqml import GKPBridge
from photographiqml.lowering import check_photographiq_contract

print(check_photographiq_contract())
bridge = GKPBridge(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
assert np.isclose(np.linalg.norm(bridge.encode([1, 0]).amplitudes), 1)
print(bridge.diagnostics())

## 32 · Physical GKP X measurement

**What the code does.** It prepares $|+_L\rangle$ and measures logical $X$ with a physical $p$-homodyne instrument. The analog outcome is post-selected to $p=0.2$ (`measurement_outcomes`), and the result is decoded with the `piquasso-fock` backend.

**Maths.** Logical $X$ on GKP is read from the momentum quadrature: $|\pm_L\rangle$ have peaks at $p\in 2\sqrt\pi\,\mathbb{Z}$ and $p\in\sqrt\pi(2\mathbb{Z}+1)$ respectively. *Hard modular decoding* rounds to the nearest lattice point:
$$b = \Big\lfloor \tfrac{p}{\sqrt\pi} + \tfrac12\Big\rfloor \bmod 2 .$$
The record stores the raw analog value $0.2$, the bit $b$, and `confidence=None`, because a hard decoder reports no posterior.

In [ ]:
import photographiq as pg

code = pg.GKPCode(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
pattern = pg.Pattern(inputs=(0,))
pattern.append(pg.Measure(0, code.logical_measurement("X"), "readout"))
result = pg.simulate(
    pattern,
    inputs={0: code.plus()},
    cutoff=24,
    backend="piquasso-fock",
    measurement_outcomes={"readout": 0.2},
)
record = result.records["readout"]
assert record.raw_outcome == 0.2 and record.bit in (0, 1)
assert record.confidence is None
print(record)

## 33 · Physical GKP Z measurement

**What the code does.** It is the same as tutorial 32 but in the computational basis. It prepares $|0_L\rangle$, measures logical $Z$ via $q$-homodyne, and conditions on $q=0.2$.

**Maths.** $|0_L\rangle$ and $|1_L\rangle$ have position peaks at $q\in2\sqrt\pi\mathbb{Z}$ and $q\in\sqrt\pi(2\mathbb{Z}+1)$. The decoder computes $b=\mathrm{round}(q/\sqrt\pi)\bmod2$. A shift $|q - q_{\text{lattice}}| < \sqrt\pi/2$ is corrected, so $0.2$ decodes as $0$ unless finite-width tails dominate.
$Z$ is an output measurement, not an intermediate MuTA XY angle.

In [ ]:
import photographiq as pg

code = pg.GKPCode(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
pattern = pg.Pattern(inputs=(0,))
pattern.append(pg.Measure(0, code.logical_measurement("Z"), "readout"))
result = pg.simulate(
    pattern,
    inputs={0: code.zero()},
    cutoff=24,
    backend="piquasso-fock",
    measurement_outcomes={"readout": 0.2},
)
record = result.records["readout"]
assert record.raw_outcome == 0.2 and record.bit in (0, 1)
assert record.confidence is None
print(record)

## 34 · Restricted physical MuTA (lowering)

**What the code does.** It builds a one-wire `PhysicalMuTA` and draws physically supported angles. It *lowers* the logical model to an executable PhotoGraphiQ `Pattern` of GKP preparations, CZs, and homodyne measurements, validates it, and prints the audit.

**Algorithm.** Lowering means:
1. **Audit** every bound angle. Only $\alpha\in\{0,\pi\}\pmod{2\pi}$ is physically supported, because these are the logical $X$-basis measurements, which equal $p$-homodyne up to relabelling.
2. Map each graph node to a GKP mode and each edge to a physical $CZ = e^{i\hat q_a\hat q_b}$.
3. Map each measurement to a homodyne instrument and a decoder signal, and record how Pauli-frame dependencies route between them.

The audit reports the peak number of live modes, the Hilbert-space dimension $\text{cutoff}^{\text{modes}}$, and the CZ count.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
from photographiqml import GKPPhysicalConfig, PhysicalMuTA, lower_muta_to_gkp

config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
model = PhysicalMuTA(1, physical_config=config)
angles = model.initialize(seed=4)
lowering = lower_muta_to_gkp(model, angles, config=config)
lowering.pattern.validate()
assert lowering.audit["supported"]
print(angles)
print({k: lowering.audit[k] for k in ("peak_live_modes", "hilbert_dimension", "cz_operations")})

## 35 · Zero-angle physical MuTA

**What the code does.** It runs the physical pattern with all angles at $0$ (logical identity). It conditions on every analog homodyne outcome being $0.0$, then prints the decoded output probabilities and the code-subspace leakage.

**Maths.** For a fixed outcome record $\mathbf{m}$, the conditional (unnormalised) output is $\tilde\rho_{\mathbf m}=K_{\mathbf m}\rho K_{\mathbf m}^\dagger$, where $K_{\mathbf m}$ is the product of homodyne projections. The final logical readout is the POVM $\{E_b\}$ given by the decoder bins:
$$P(b\,|\,\mathbf m) = \frac{\mathrm{Tr}[E_b\tilde\rho_{\mathbf m}]}{\mathrm{Tr}\,\tilde\rho_{\mathbf m}} .$$
Ideally $P(0)=1$. Finite $\Delta$ makes peaks overlap decoder boundaries, so a logical identity still shows decoded errors. *Leakage* $=1-\mathrm{Tr}[\Pi_{\text{code}}\rho]$ measures weight outside the finite code span.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
from photographiqml import GKPPhysicalConfig, PhysicalMuTA

config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
model = PhysicalMuTA(1, physical_config=config)
result = model.run(
    [1, 0], mode="physical-conditional", analog_outcomes=dict.fromkeys(model.measurement_order, 0.0)
)
assert result.representation == "gkp-physical"
assert not result.convergence["certified"]
print(result.decoded_joint_probabilities)
print(result.diagnostics["marginal_code_subspace_leakage"])

## 36 · Pauli-frame propagation

**What the code does.** It supplies made-up decoded bits $s_v\in\{0,1\}$ for every measured node and computes the Pauli frame $X^{x}Z^{z}$ accumulated on the output node. It then checks the readout correction rules.

**Maths.** In MBQC every outcome $s_v$ produces a by-product: on the flow successor $f(v)$, $X^{s_v}$, and on the neighbours of $f(v)$, $Z^{s_v}$. These combine into a frame
$$x=\bigoplus_{v} s_v\,[\text{X-dep}],\qquad z=\bigoplus_v s_v\,[\text{Z-dep}] .$$
A Pauli measurement $P$ must be flipped when the frame *anticommutes* with $P$. $X$ anticommutes with $Z$, so the correction for an $X$ readout is $z$, and for a $Z$ readout it is $x$.
Frames are classical bookkeeping. No physical displacement is applied.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
from photographiqml import GKPPhysicalConfig, PhysicalMuTA
from photographiqml.lowering import node_frame

config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
model = PhysicalMuTA(1, physical_config=config)
records = {("bit", v): int(i % 2) for i, v in enumerate(model.measurement_order)}
frame = node_frame(model, model.output_nodes[0], records)
assert frame.correction("X") == frame.z_bit
assert frame.correction("Z") == frame.x_bit
print(frame)

## 37 · Hard vs soft decoding

**What the code does.** It decodes the same analog value $q=0.2$ with a hard decoder (a single bit, no confidence) and with a soft decoder (a posterior over $\{0,1\}$).

**Maths.** The soft decoder uses the finite-GKP likelihoods $p(q\,|\,\mu)=|\langle q|\mu_L\rangle|^2$ with equal priors:
$$P(\mu\,|\,q) = \frac{p(q|\mu)}{p(q|0)+p(q|1)},\qquad \text{confidence}=\max_\mu P(\mu|q).$$
The posterior is well calibrated only for this preparation ensemble ($|0_L\rangle$ vs $|1_L\rangle$ with equal priors). It is not calibrated for arbitrary intermediate MuTA states.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
from photographiqml import GKPBridge

bridge = GKPBridge(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
hard = bridge.resource_readout().decoder.decode(0.2)
soft = bridge.resource_readout(decoder="soft").decoder.decode(0.2)
assert hard.confidence is None and soft.confidence is not None
assert abs(sum(soft.probabilities) - 1) < 1e-12
print("hard confidence:", hard.confidence)
print("preparation posterior:", soft.probabilities)

## 38 · Multi-mode joint readout

**What the code does.** It runs a two-wire physical MuTA from $|00\rangle$ on one conditional trajectory and obtains the joint decoded distribution $P(b_0,b_1)$. It checks that the distribution is not the product of its marginals.

**Maths.** The joint POVM $E_{b_0}\otimes E_{b_1}$ acts on the full two-mode density matrix:
$$P(b_0,b_1)=\mathrm{Tr}\big[(E_{b_0}\otimes E_{b_1})\rho\big].$$
The *connected* probability
$$C = P(0,0) - P_0(0)\,P_1(0)$$
is zero for uncorrelated outputs. A nonzero $C$ shows that the joint distribution must come from $\rho$, not from separate marginals.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
from photographiqml import GKPPhysicalConfig, PhysicalMuTA

config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
model = PhysicalMuTA(1, physical_config=config)
model = PhysicalMuTA(2, physical_config=config)
result = model.run(
    [1, 0, 0, 0],
    mode="physical-conditional",
    analog_outcomes=dict.fromkeys(model.measurement_order, 0.0),
)
p = result.decoded_joint_probabilities
assert len(p) == 4 and abs(sum(p.values()) - 1) < 1e-9
marginals = list(result.decoded_marginals.values())
connected = p[(0, 0)] - marginals[0][0] * marginals[1][0]
assert abs(connected) > 1e-5
print(p)
print("connected probability:", connected)

## 39 · Physical shots and statistical error

**What the code does.** It runs 3 stochastic shots. Each shot samples a full trajectory of homodyne outcomes and then samples output bits. It reports the averaged conditional POVM probabilities, their standard errors, and the empirical bit frequencies.

**Maths.** For shot $k$ with outcome record $\mathbf m_k$, let $\hat P_k(b) = P(b|\mathbf m_k)$. There are two estimators:
$$\bar P(b) = \frac1K\sum_k \hat P_k(b),\qquad \mathrm{SE}=\frac{s}{\sqrt K},\qquad \hat f(b) = \frac{\#\{k: b_k=b\}}{K}.$$
$\bar P$ (Rao–Blackwellised) has lower variance than the raw frequency $\hat f$. Both converge to the unconditional $P(b)$ as $K\to\infty$. Three shots only exercise the API.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
from photographiqml import GKPPhysicalConfig, PhysicalMuTA

config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
model = PhysicalMuTA(1, physical_config=config)
result = model.run([1, 0], shots=3, seed=14)
assert len(result.sampled_output_bits) == 3
assert result.standard_errors is not None
print("mean conditional POVM:", result.decoded_joint_probabilities)
print("standard errors:", result.standard_errors)
print("sampled bit frequencies:", result.empirical_probabilities)

## 40 · Logical vs physical comparison

**What the code does.** It compares the physical conditional result from tutorial 35 with the ideal logical prediction.

**Maths.** A state fidelity is undefined, because the qubit space $\mathbb{C}^2$ and the Fock space $\mathbb{C}^{24}$ differ (`finite_state_fidelity = None`). The comparison is therefore made on decoded distributions:
$$\mathrm{TVD}(P,Q)=\tfrac12\sum_b |P(b)-Q(b)| \in [0,1],$$
together with observable differences $\langle Z\rangle_{\text{phys}}-\langle Z\rangle_{\text{log}}$. For one bit, $\Delta\langle Z\rangle = -2\,\mathrm{TVD}$ when the physical result errs toward 1.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
from photographiqml import GKPPhysicalConfig, PhysicalMuTA, compare_logical_physical

config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
model = PhysicalMuTA(1, physical_config=config)
result = model.run(
    [1, 0], mode="physical-conditional", analog_outcomes=dict.fromkeys(model.measurement_order, 0.0)
)
comparison = compare_logical_physical(result)
assert comparison["finite_state_fidelity"] is None
assert comparison["total_variation_distance"] >= 0
print(comparison)

## 41 · Physical convergence

**What the code does.** It reruns the same conditional trajectory on quadrature grids of 513 and 1025 points. It reports the largest change in decoded probabilities between consecutive refinements.

**Maths.** A numerical refinement study along axis $h$ (here the grid spacing) uses
$$\delta_j = \max_b\big|P_{h_j}(b)-P_{h_{j-1}}(b)\big| .$$
A small $\delta$ only shows that the *quadrature integration* has converged. It says nothing about the Fock cutoff, the number of peaks, or the physics of finite squeezing ($\Delta$). Those are separate axes, which is why `certified` stays `False`.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
from photographiqml import GKPPhysicalConfig, PhysicalMuTA

config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
model = PhysicalMuTA(1, physical_config=config)
study = model.physical_convergence(
    [1, 0],
    [513, 1025],
    axis="grid_points",
    mode="physical-conditional",
    analog_outcomes=dict.fromkeys(model.measurement_order, 0.0),
)
assert not study["certified"]
print(study["axis_type"])
print([row["max_probability_delta"] for row in study["rows"]])

## 42 · Unsupported arbitrary XY angles: why they fail

**What the code does.** It tries to run a physical MuTA with $\alpha=0.37$ rad and prints the explicit `NotImplementedError`.

**Why.** A logical XY measurement $\cos\alpha\,X+\sin\alpha\,Y$ is **not** a rotated homodyne quadrature for GKP. Rotating the homodyne angle by $\alpha$ measures $\cos\alpha\,\hat q+\sin\alpha\,\hat p$, whose lattice is not the GKP stabiliser lattice, so modular decoding does not yield the logical eigenvalue. For non-Clifford $\alpha$, a logical $e^{i\alpha Z/2}$ requires magic-state injection and a gate-teleportation protocol.
The library therefore rejects the request *before* any simulation and does not return a misleading result.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
from photographiqml import GKPPhysicalConfig, PhysicalMuTA

config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
model = PhysicalMuTA(1, physical_config=config)
try:
    model.run([1, 0], {"alpha.w0.c1": 0.37})
except NotImplementedError as error:
    print(error)
else:
    raise AssertionError("Unsupported physical measurement executed")

## 43 · Logical training → physical validation

**What the code does.** It trains a logical MuTA for 2 Adam steps toward $P(1)=0.3$. It audits the trained angles for physical support and shows that `GKPBridge.run` refuses the continuous angles. They are not rounded silently.

**Maths.** The loss is $\mathcal{L}(\theta)=\big(P_\theta(1)-0.3\big)^2$. The trained $\theta$ is continuous, but the physical family is $\{0,\pi\}^P$. Rounding to that family would change the model in a way the optimizer never saw, so the library refuses and states the reason.

In [ ]:
import numpy as np

from photographiqml import GKPBridge, MuTA, Trainer

model = MuTA(1)
parameters, history = Trainer(epochs=2).fit(
    lambda p: float((model.run([1, 0], p).probabilities[1] - 0.3) ** 2), np.full(4, 0.5)
)
audit = model.physical_capabilities(parameters)
assert not audit["supported"]
try:
    GKPBridge().run(model, [1, 0], parameters)
except NotImplementedError as error:
    print(error)
else:
    raise AssertionError("Continuous candidate was silently altered")

## 44 · Discrete physical-angle search

**What the code does.** It optimises directly over the physically supported discrete family $\theta\in\{0,\pi\}^4$, using a noisy 1-shot physical objective $P(\text{output}=1)$.

**Algorithm (coordinate search).** Start from $\theta=0$. For each coordinate $k$ (one sweep), flip $\theta_k\leftrightarrow\theta_k+\pi$, evaluate the objective, and keep the flip if it lowers the loss.
Evaluations $=1 + P\cdot\text{sweeps} = 1+4 = 5$. Every evaluation runs a valid finite-GKP pattern, so no candidate ever leaves the supported set.
**Caveat.** With 1 shot the objective has sampling noise, and $\{0,\pi\}$ flips only relabel Paulis, so the family has very limited expressivity.

**Shared physical configuration.** `GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)` means:
a Fock truncation $n \le 23$; each GKP peak is a Gaussian of width $\Delta = 0.9$; the comb is damped by a Gaussian envelope of width $\kappa^{-1}$ with $\kappa = 0.9$;
$\pm 4$ peaks are kept per codeword; and quadrature integrals are evaluated on a 1025-point grid. These are *broad, low-squeezing* codewords chosen so the
simulation runs quickly, so expect visible logical errors.

In [ ]:
import numpy as np

from photographiqml import GKPPhysicalConfig, PhysicalMuTA
from photographiqml.physical_training import DiscreteSearch

config = GKPPhysicalConfig(cutoff=24, peak_width=0.9, envelope=0.9, peaks=4, grid_points=1025)
model = PhysicalMuTA(1, physical_config=config)


def objective(angles):
    result = model.run([1, 0], angles, shots=1, seed=14)
    return result.decoded_joint_probabilities[(1,)]


search = DiscreteSearch(sweeps=1, seed=14).fit(model, objective, initial=np.zeros(4))
assert set(search.parameters) <= {0, np.pi}
assert len(search.evaluations) == 5
print("selected:", search.parameters)
print("estimated error:", search.loss)